# Moduł 6: Wstęp do uczenia maszynowego

**Uczenie maszynowe (ML)** to dziedzina AI, w której komputer **uczy się z danych** zamiast być programowany jawnie.

## Spis treści
1. Typy uczenia
2. Podział danych: train / val / test
3. Metryki ewaluacji
4. Overfitting i underfitting
5. Walidacja krzyżowa (cross-validation)
6. Przygotowanie danych
7. Redukcja wymiarowości (PCA)
8. K-Nearest Neighbors (KNN)
9. K-Means Clustering
10. t-SNE i UMAP
11. DBSCAN, Hierarchical i Spectral Clustering
12. **Zadania**

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
 accuracy_score, precision_score, recall_score, f1_score,
 confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.datasets import load_iris, make_classification
sns.set_style("whitegrid")

## 1. Typy uczenia maszynowego

### Uczenie nadzorowane (supervised)
- Mamy zbiór danych $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^{n}$ — pary (dane, etykieta)
- Model uczy się funkcji $f: \mathcal{X} \to \mathcal{Y}$ takiej, że $f(x) \approx y$
- **Klasyfikacja:** $y \in \{1, 2, \dots, K\}$ — klasy dyskretne (kot/pies, spam/nie-spam)
- **Regresja:** $y \in \mathbb{R}$ — wartości ciągłe (cena domu, temperatura)

### Uczenie nienadzorowane (unsupervised)
- Tylko dane $\mathcal{D} = \{x_i\}_{i=1}^{n}$, **bez etykiet**
- Model szuka struktury/wzorców w danych
- **Klasteryzacja:** partycja zbioru na grupy $C_1, C_2, \dots, C_K$ minimalizująca odległości wewnątrz-grupowe
- **Redukcja wymiarów:** odwzorowanie $f: \mathbb{R}^d \to \mathbb{R}^k$ gdzie $k \ll d$, zachowujące strukturę danych

### Uczenie ze wzmocnieniem (reinforcement learning)
- Agent podejmuje akcje $a_t$ w stanie $s_t$, przechodząc do stanu $s_{t+1}$
- Otrzymuje nagrodę $r_t$ za każdą akcję
- Cel: znaleźć politykę $\pi(a|s)$ maksymalizującą **skumulowaną zdyskontowaną nagrodę**:

$$G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}$$

gdzie $\gamma \in [0, 1]$ to współczynnik dyskonta (jak bardzo cenimy przyszłe nagrody).

## 2. Podział danych: Train / Validation / Test

**Dlaczego dzielimy?** Bo chcemy sprawdzić, czy model **uogólnia** na nowe, niewidziane dane.

| Zbiór | Cel | Typowy % |
|-------|-----|----------|
| **Train** | Uczenie modelu | 60-80% |
| **Validation** | Dobór hiperparametrów | 10-20% |
| **Test** | Finalna ocena | 10-20% |

**ZŁOTA ZASADA:** Zbiór testowy oglądasz **TYLKO RAZ**, na samym końcu!

In [ ]:
# Przykład podziału danych
iris = load_iris()
X, y = iris.data, iris.target

# Podział na train+val i test
X_trainval, X_test, y_trainval, y_test = train_test_split(
 X, y, test_size=0.2, random_state=42, stratify=y
)
# Podział train+val na train i val
X_train, X_val, y_train, y_val = train_test_split(
 X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Val: {len(X_val)} ({len(X_val)/len(X)*100:.0f}%)")
print(f"Test: {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")

# stratify=y zapewnia, że proporcje klas są zachowane w każdym zbiorze

## 3. Metryki ewaluacji

### Macierz pomyłek (Confusion Matrix)

Dla klasyfikacji binarnej (klasa pozytywna vs negatywna):

| | Predicted Positive | Predicted Negative |
|--|-------------------|-------------------|
| **Actual Positive** | TP (True Positive) | FN (False Negative) |
| **Actual Negative** | FP (False Positive) | TN (True Negative) |

### Kluczowe metryki

**Accuracy** — odsetek poprawnych predykcji:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

**Precision** — z tych, które model oznaczył jako pozytywne, ile jest faktycznie pozytywnych:

$$\text{Precision} = \frac{TP}{TP + FP}$$

**Recall (Sensitivity/Czułość)** — z faktycznie pozytywnych, ile model złapał:

$$\text{Recall} = \frac{TP}{TP + FN}$$

**F1-score** — średnia harmoniczna precision i recall:

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}$$

**F-beta score** — uogólnienie F1, gdzie $\beta$ kontroluje wagę recall vs precision:

$$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \cdot \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}$$

- $\beta = 1$ → F1 (równowaga precision i recall)
- $\beta = 2$ → F2 (recall ważniejszy — np. diagnostyka medyczna)
- $\beta = 0.5$ → F0.5 (precision ważniejszy — np. filtr spamu)

**Balanced Accuracy** — średnia recall z każdej klasy (odporna na niezbalansowane klasy):

$$\text{Balanced Accuracy} = \frac{1}{K}\sum_{k=1}^{K} \text{Recall}_k$$

**ROC AUC** — pole pod krzywą ROC (Receiver Operating Characteristic):
- Krzywa ROC rysuje $\text{TPR}$ (True Positive Rate = Recall) vs $\text{FPR}$ (False Positive Rate = $\frac{FP}{FP+TN}$) dla różnych progów klasyfikacji
- AUC $\in [0.5, 1.0]$ — 0.5 = losowy klasyfikator, 1.0 = idealny
- **Interpretacja probabilistyczna:** $P(\text{score}(x^+) > \text{score}(x^-))$ — prawdopodobieństwo, że model przypisze wyższy score pozytywnej próbce niż negatywnej

### Kiedy co?
- **Accuracy** — gdy klasy są zbalansowane
- **Precision** — gdy FP jest kosztowny (np. filtr spamu — nie chcesz usuwać ważnych maili)
- **Recall** — gdy FN jest kosztowny (np. diagnostyka raka — nie chcesz pominąć chorego)
- **F1** — kompromis między precision i recall
- **Balanced Accuracy** — niezbalansowane klasy (np. 95% klasy 0, 5% klasy 1)
- **ROC AUC** — ocena ogólnej jakości rankingu (niezależna od progu)

In [ ]:
# Trening i ewaluacja KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_val)

# Metryki
print("=== METRYKI ===")
print(f"Accuracy: {accuracy_score(y_val, y_pred):.3f}")
print(f"Precision: {precision_score(y_val, y_pred, average='macro'):.3f}")
print(f"Recall: {recall_score(y_val, y_pred, average='macro'):.3f}")
print(f"F1-score: {f1_score(y_val, y_pred, average='macro'):.3f}")

# Pełny raport
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_val, y_pred, target_names=iris.target_names))

# Macierz pomyłek
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
 xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Macierz pomyłek")
plt.show()

## 4. Overfitting vs Underfitting

| Problem | Opis | Objawy |
|---------|------|--------|
| **Underfitting** | Model za prosty | Słaby wynik na train I test |
| **Overfitting** | Model "zapamiętał" dane | Świetny train, słaby test |
| **Just right** | Model uogólnia | Dobry wynik na obu |

### Bias-Variance Tradeoff — dekompozycja błędu

Błąd generalizacji modelu można rozłożyć na trzy składniki:

$$\text{MSE} = \mathbb{E}\left[(y - \hat{f}(x))^2\right] = \underbrace{\text{Bias}^2(\hat{f})}_{\text{obciążenie}} + \underbrace{\text{Var}(\hat{f})}_{\text{wariancja}} + \underbrace{\sigma^2}_{\text{szum}}$$

Gdzie:
- **Bias** $= \mathbb{E}[\hat{f}(x)] - f(x)$ — średnie odchylenie predykcji od prawdziwej wartości. Wysoki bias = underfitting (model zbyt prosty, nie oddaje złożoności danych).
- **Variance** $= \mathbb{E}\left[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2\right]$ — jak bardzo predykcja zmienia się zależnie od konkretnego zbioru treningowego. Wysoka wariancja = overfitting (model nadmiernie dopasowuje się do danych).
- **$\sigma^2$** — nieredukowalny szum w danych (nie da się go wyeliminować).

**Trade-off:** Zmniejszenie bias (bardziej złożony model) → zwiększenie wariancji i odwrotnie. Optymalny model balansuje oba.

### Regularyzacja — matematyczne ograniczenie złożoności

Dodajemy **karę za złożoność** do funkcji kosztu:

$$\mathcal{L}_{\text{reg}}(\theta) = \mathcal{L}(\theta) + \lambda \cdot \Omega(\theta)$$

| Typ | Kara $\Omega(\theta)$ | Efekt |
|-----|----------------------|-------|
| **L2 (Ridge)** | $\|\theta\|_2^2 = \sum \theta_i^2$ | Zmniejsza wagi (nie do zera) |
| **L1 (Lasso)** | $\|\theta\|_1 = \sum |\theta_i|$ | Zeruje nieważne wagi (selekcja cech!) |
| **Elastic Net** | $\alpha\|\theta\|_1 + (1-\alpha)\|\theta\|_2^2$ | Kombinacja obu |

$\lambda$ — siła regularyzacji: $\lambda = 0$ → brak regularyzacji, $\lambda \to \infty$ → model trywialny.

### Jak zapobiegać overfittingowi?
1. Więcej danych treningowych
2. Regularyzacja (L1, L2, Elastic Net)
3. Walidacja krzyżowa
4. Dropout (w sieciach neuronowych)
5. Early stopping
6. Data augmentation

In [ ]:
# Demonstracja overfitting/underfitting na KNN
# k=1: overfit (zapamiętuje każdy punkt)
# k=50: underfit (za dużo uśredniania)

k_values = range(1, 31)
train_scores = []
val_scores = []

for k in k_values:
 knn = KNeighborsClassifier(n_neighbors=k)
 knn.fit(X_train, y_train)
 train_scores.append(knn.score(X_train, y_train))
 val_scores.append(knn.score(X_val, y_val))

plt.figure(figsize=(10, 5))
plt.plot(k_values, train_scores, 'o-', label="Train")
plt.plot(k_values, val_scores, 's-', label="Validation")
plt.xlabel("k (liczba sąsiadów)")
plt.ylabel("Accuracy")
plt.title("Overfitting vs Underfitting (KNN)")
plt.legend()
plt.grid(True)
plt.show()
# k=1: train=100% ale val niższe → OVERFIT
# Duże k: obie linie spadają → UNDERFIT

## 5. Walidacja krzyżowa (Cross-Validation)

Zamiast jednego podziału train/val, dzielimy dane na **k foldów** i trenujemy k razy:

```
Fold 1: [VAL][TRAIN][TRAIN][TRAIN][TRAIN]
Fold 2: [TRAIN][VAL][TRAIN][TRAIN][TRAIN]
Fold 3: [TRAIN][TRAIN][VAL][TRAIN][TRAIN]
Fold 4: [TRAIN][TRAIN][TRAIN][VAL][TRAIN]
Fold 5: [TRAIN][TRAIN][TRAIN][TRAIN][VAL]
```

**Wynik k-fold CV:**

$$\text{CV Score} = \frac{1}{k} \sum_{i=1}^{k} \text{Score}_i$$

$$\text{Odchylenie std} = \sqrt{\frac{1}{k-1} \sum_{i=1}^{k} (\text{Score}_i - \overline{\text{Score}})^2}$$

**Stratified k-fold** — każdy fold zachowuje proporcje klas (ważne przy niezbalansowanych danych!)

**Leave-One-Out (LOO)** — skrajny przypadek: $k = n$ (każda próbka to osobny fold). Niskie bias, ale wysokie variance i wolne.

In [ ]:
# Cross-validation w scikit-learn
knn = KNeighborsClassifier(n_neighbors=5)

# 5-fold cross-validation
scores = cross_val_score(knn, X_trainval, y_trainval, cv=5, scoring="accuracy")
print(f"Scores per fold: {scores}")
print(f"Mean: {scores.mean():.3f} ± {scores.std():.3f}")

# Porównanie różnych k z cross-validation
k_values = range(1, 21)
cv_means = []
cv_stds = []
for k in k_values:
 knn = KNeighborsClassifier(n_neighbors=k)
 scores = cross_val_score(knn, X_trainval, y_trainval, cv=5)
 cv_means.append(scores.mean())
 cv_stds.append(scores.std())

plt.figure(figsize=(10, 5))
plt.errorbar(k_values, cv_means, yerr=cv_stds, fmt='o-', capsize=3)
plt.xlabel("k")
plt.ylabel("CV Accuracy")
plt.title("Dobór k za pomocą Cross-Validation")
plt.show()
best_k = list(k_values)[np.argmax(cv_means)]
print(f"Najlepsze k = {best_k}")

## 6. Przygotowanie danych (preprocessing)

### Standaryzacja (z-score)

$$x_{\text{std}} = \frac{x - \mu}{\sigma}$$

Po standaryzacji: średnia = 0, odchylenie standardowe = 1.

**Dlaczego?** Algorytmy oparte na odległościach (KNN, SVM) i sieci neuronowe wymagają danych w podobnych skalach.

### Normalizacja Min-Max

Alternatywnie, skalowanie do zakresu $[0, 1]$:

$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

Ogólniej, do zakresu $[a, b]$:

$$x_{\text{scaled}} = a + \frac{(x - x_{\min})(b - a)}{x_{\max} - x_{\min}}$$

### Robust Scaler

Odporny na outliery — używa mediany i IQR (interquartile range):

$$x_{\text{robust}} = \frac{x - \text{median}(x)}{Q_3 - Q_1}$$

### Data Leakage — krytyczne pojęcie

**Data leakage** (wyciek danych) to sytuacja, gdy informacje z danych testowych "przeciekają" do procesu uczenia.

Przykłady wycieku:
- `scaler.fit()` na **całości** danych (zamiast tylko na train)
- Feature selection na całości danych
- Imputacja braków na całości danych

**Reguła:** Wszelkie transformacje `fit()` **tylko na danych treningowych**, potem `transform()` na walidacji/teście.

$$\text{Pipeline:} \quad X_{\text{train}} \xrightarrow{\text{fit\_transform}} X'_{\text{train}} \quad | \quad X_{\text{test}} \xrightarrow{\text{transform}} X'_{\text{test}}$$

In [ ]:
# Standaryzacja
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # fit + transform na TRAIN
X_val_scaled = scaler.transform(X_val) # TYLKO transform na val!
X_test_scaled = scaler.transform(X_test) # TYLKO transform na test!

print(f"Przed: średnie = {X_train.mean(axis=0).round(2)}")
print(f"Po: średnie = {X_train_scaled.mean(axis=0).round(2)}")
print(f"Po: std = {X_train_scaled.std(axis=0).round(2)}")

# UWAGA: scaler.fit() tylko na danych treningowych!
# W przeciwnym razie mamy "data leakage" (wyciek informacji z testu)

## 7. Redukcja wymiarowości — PCA

**PCA (Principal Component Analysis)** znajduje kierunki o największej wariancji w danych i rzutuje na nie.

### Algorytm PCA krok po kroku

1. **Standaryzuj dane** — wycentruj (odejmij średnią) i opcjonalnie skaluj
2. **Oblicz macierz kowariancji:**

$$\Sigma = \frac{1}{n-1} X^T X \in \mathbb{R}^{d \times d}$$

gdzie $X \in \mathbb{R}^{n \times d}$ to wycentrowana macierz danych.

3. **Rozwiąż zagadnienie własne** (eigendecomposition):

$$\Sigma \mathbf{v}_i = \lambda_i \mathbf{v}_i$$

gdzie $\lambda_i$ — wartość własna (eigenvalue), $\mathbf{v}_i$ — wektor własny (eigenvector).

4. **Posortuj wektory własne** malejąco wg wartości własnych: $\lambda_1 \geq \lambda_2 \geq \ldots \geq \lambda_d$

5. **Weź $k$ pierwszych wektorów własnych** jako macierz projekcji $W_k = [\mathbf{v}_1 | \mathbf{v}_2 | \ldots | \mathbf{v}_k] \in \mathbb{R}^{d \times k}$

6. **Rzutuj dane:**

$$Z = X \cdot W_k \in \mathbb{R}^{n \times k}$$

### Explained Variance Ratio

Procent wariancji zachowany przez $k$-ty komponent:

$$\text{EVR}_k = \frac{\lambda_k}{\sum_{i=1}^{d} \lambda_i}$$

**Kryterium wyboru $k$**: zachowaj tyle komponentów, by suma EVR $\geq$ 95%:

$$\sum_{i=1}^{k} \text{EVR}_i \geq 0.95$$

### Interpretacja geometryczna

- PC1 = kierunek **największej wariancji** w danych
- PC2 = kierunek największej wariancji **ortogonalny** do PC1
- Każdy PCi jest kombinacją liniową oryginalnych cech: $\text{PC}_i = \sum_{j=1}^{d} v_{ij} \cdot x_j$

### Zastosowania
- Wizualizacja danych wielowymiarowych (rzut na 2D/3D)
- Usunięcie szumu (odrzucenie komponentów o małej wariancji)
- Przyspieszenie treningu (mniej wymiarów → mniej operacji)
- Dekorelacja cech (składowe główne są nieskorelowane)

### Alternatywy dla PCA
| Metoda | Typ | Zastosowanie |
|--------|-----|-------------|
| **t-SNE** | Nieliniowa | Wizualizacja 2D/3D (nie do redukcji przed modelem!) |
| **UMAP** | Nieliniowa | Wizualizacja + szybsza niż t-SNE |
| **LDA** | Liniowa, nadzorowana | Redukcja z uwzględnieniem klas |
| **Autoenkoder** | Nieliniowa, deep learning | Nieliniowa redukcja wymiarów |

In [ ]:
# PCA: redukcja 4D → 2D (Iris)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_trainval)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Suma: {pca.explained_variance_ratio_.sum():.3f}")
# Ile informacji zachowaliśmy? 

# Wizualizacja
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_trainval, 
 cmap="viridis", alpha=0.7)
plt.colorbar(scatter, label="Klasa")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% wariancji)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% wariancji)")
plt.title("Iris — PCA 2D")
plt.show()

---

## 8. K-Nearest Neighbors (KNN)

**KNN** to algorytm **leniwego uczenia** — nie buduje jawnego modelu, a predykcja opiera się na $k$ najbliższych sąsiadach w zbiorze treningowym.

### Algorytm

Dla nowego punktu $x$:
1. Oblicz odległość do **wszystkich** punktów treningowych
2. Wybierz $k$ najbliższych sąsiadów
3. **Klasyfikacja**: głosowanie większościowe (najczęstsza klasa wśród sąsiadów)
4. **Regresja**: średnia wartości sąsiadów

### Odległość

Najczęściej euklidesowa:

$$d(x, x') = \sqrt{\sum_{i=1}^{d} (x_i - x'_i)^2} = \|x - x'\|_2$$

Inne metryki:
- **Manhattan** (L1): $d(x, x') = \sum |x_i - x'_i|$
- **Minkowski** (Lp): $d(x, x') = \left(\sum |x_i - x'_i|^p\right)^{1/p}$
- **Cosine distance**: $d(x, x') = 1 - \frac{x^T x'}{\|x\| \|x'\|}$

### Wybór $k$

| Wartość $k$ | Efekt |
|-------------|-------|
| Małe $k$ (np. 1-3) | Niska bias, wysoka wariancja → **overfitting** |
| Duże $k$ (np. 50+) | Wysoka bias, niska wariancja → **underfitting** |

Optymalny $k$ dobieramy przez **cross-validation**.

### Standaryzacja jest kluczowa!

KNN jest **wrażliwy na skalę cech**. Cecha o zakresie [0, 1000] zdominuje cechę [0, 1] w odległości euklidesowej. **Zawsze standaryzuj dane** przed KNN.

### Weighted KNN

Zamiast prostego głosowania, waga sąsiada = $\frac{1}{d(x, x_i)}$ — bliżsi sąsiedzi mają większy wpływ.

### Złożoność

- **Trening**: $O(1)$ — tylko zapamiętaj dane!
- **Predykcja**: $O(n \cdot d)$ na punkt — wolne dla dużych zbiorów
- Przyspieszenie: KD-tree, Ball-tree (ale działają źle w wysokich wymiarach)

In [ ]:
# KNN — demonstracja i wpływ k
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=500, n_features=10, n_informative=5, 
 random_state=42, n_classes=3)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Porównanie k
print("=== Wpływ k na accuracy (5-fold CV) ===")
for k in [1, 3, 5, 10, 20, 50]:
 knn = KNeighborsClassifier(n_neighbors=k)
 scores = cross_val_score(knn, X_scaled, y, cv=5, scoring='accuracy')
 print(f" k={k:2d} | accuracy = {scores.mean():.4f} ± {scores.std():.4f}")

# Wpływ standardyzacji
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_scaled = KNeighborsClassifier(n_neighbors=5)
raw_scores = cross_val_score(knn_raw, X, y, cv=5)
scaled_scores = cross_val_score(knn_scaled, X_scaled, y, cv=5)
print(f"\nBez standaryzacji: {raw_scores.mean():.4f}")
print(f"Ze standaryzacją: {scaled_scores.mean():.4f}")
print(" Standaryzacja prawie zawsze pomaga w KNN!")

---

## 9. K-Means Clustering

**K-Means** to najpopularniejszy algorytm **klasteryzacji** — dzieli dane na $k$ skupień (klastrów) minimalizując wewnętrzną wariancję.

### Algorytm (Lloyd's)

1. **Inicjalizacja**: wybierz $k$ centroidów $\mu_1, \ldots, \mu_k$ (np. losowo z danych)
2. **Krok E (assignment)**: przypisz każdy punkt do najbliższego centroidu:
$$c_i = \arg\min_j \|x_i - \mu_j\|^2$$
3. **Krok M (update)**: przelicz centroidy jako średnią przypisanych punktów:
$$\mu_j = \frac{1}{|C_j|} \sum_{x_i \in C_j} x_i$$
4. Powtarzaj kroki 2-3, aż centroidy się ustabilizują (lub max iteracji)

### Funkcja celu — Inertia (WCSS)

$$J = \sum_{j=1}^{k} \sum_{x_i \in C_j} \|x_i - \mu_j\|^2$$

K-Means minimalizuje **Within-Cluster Sum of Squares** (WCSS, inertia).

### K-Means++ (lepsza inicjalizacja)

Losowa inicjalizacja może prowadzić do złych rozwiązań. K-Means++ wybiera centroidy tak, by były **rozproszone**:
1. Wybierz pierwszy centroid losowo
2. Kolejne centroidy: prawdopodobieństwo wyboru $\propto d^2$ (odległość od najbliższego centroidu)
3. Scikit-learn używa K-Means++ domyślnie (`init='k-means++'`)

### Metoda łokcia (Elbow Method)

Wybór $k$ — rysuj **inertia** vs. $k$. Szukaj "łokcia" — punktu, po którym spadek inertii jest niewielki.

### Metryki oceny klasteryzacji

| Metryka | Zakres | Interpretacja | Wymaga etykiet? |
|---------|--------|--------------|----------------|
| **Silhouette Score** | [-1, 1] | Wyższy = lepszy | Nie |
| **Davies-Bouldin** | [0, ∞) | Niższy = lepszy | Nie |
| **Calinski-Harabasz** | [0, ∞) | Wyższy = lepszy | Nie |
| **Adjusted Rand Index** | [-1, 1] | 1 = idealne | Tak |

**Silhouette Score** dla punktu $i$:

$$s_i = \frac{b_i - a_i}{\max(a_i, b_i)}$$

gdzie $a_i$ = średnia odległość do punktów w tym samym klastrze, $b_i$ = średnia odległość do najbliższego innego klastra.

### Ograniczenia K-Means

- Zakłada **sferyczne** klastry o podobnym rozmiarze
- Wrażliwy na inicjalizację i outliers
- Musi znać $k$ z góry
- Nie radzi sobie z klastrami o złożonych kształtach (np. pierścienie)

In [ ]:
# K-Means — demonstracja z Elbow Method i Silhouette
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.datasets import make_blobs

# Generuj dane
X_blobs, y_true = make_blobs(n_samples=500, centers=4, cluster_std=1.0, random_state=42)

# Elbow Method
inertias = []
silhouettes = []
K_range = range(2, 10)

for k in K_range:
 km = KMeans(n_clusters=k, n_init=10, random_state=42)
 labels = km.fit_predict(X_blobs)
 inertias.append(km.inertia_)
 silhouettes.append(silhouette_score(X_blobs, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(K_range, inertias, 'bo-')
ax1.set_xlabel('k (liczba klastrów)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.set_title('Elbow Method')
ax1.grid(True, alpha=0.3)

ax2.plot(K_range, silhouettes, 'ro-')
ax2.set_xlabel('k')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score vs k')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Najlepsze k
best_k = list(K_range)[np.argmax(silhouettes)]
km_best = KMeans(n_clusters=best_k, n_init=10, random_state=42)
labels_best = km_best.fit_predict(X_blobs)

print(f"Najlepsze k (Silhouette): {best_k}")
print(f"Silhouette: {silhouette_score(X_blobs, labels_best):.4f}")
print(f"Davies-Bouldin: {davies_bouldin_score(X_blobs, labels_best):.4f}")
print(f"Calinski-Harabasz: {calinski_harabasz_score(X_blobs, labels_best):.4f}")

---

## 10. t-SNE i UMAP

Metody **nieliniowej** redukcji wymiarów do wizualizacji danych wielowymiarowych w 2D/3D.

### t-SNE (t-distributed Stochastic Neighbor Embedding)

**Idea**: zachowaj lokalne sąsiedztwo — bliskie punkty w wysokim wymiarze powinny być bliskie w 2D.

1. W przestrzeni oryginalnej: prawdopodobieństwo podobieństwa par (Gaussian):
$$p_{j|i} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|x_i - x_k\|^2 / 2\sigma_i^2)}, \quad p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}$$

2. W przestrzeni niskowymiarowej: rozkład **t-Studenta** z 1 st. swobody (grubsze ogony!):
$$q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \neq l} (1 + \|y_k - y_l\|^2)^{-1}}$$

3. Minimalizuj **KL divergencję**: $\text{KL}(P \| Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$

**Kluczowy hiperparametr: perplexity** (zwykle 5-50) — ile sąsiadów bierze pod uwagę.

### UMAP (Uniform Manifold Approximation and Projection)

**Szybsza** alternatywa dla t-SNE, oparta na teorii topologicznej:

- Buduje **graf sąsiedztwa** w wysokim wymiarze (fuzzy simplicial sets)
- Optymalizuje układ niskowymiarowy by zachować topologię grafu
- **Cross-entropy** zamiast KL divergencji

| Cecha | t-SNE | UMAP |
|-------|-------|------|
| Prędkość | Wolny ($O(n^2)$ lub $O(n \log n)$) | Szybki |
| Struktura globalna | Słabo zachowana | Lepiej zachowana |
| Deterministyczność | Nie | Nie (ale bardziej stabilny) |
| Nowe punkty | Nie da się dodać | Można (transform) |
| Metryki | Euklidesowa | Wiele dostępnych |

### Ważne zasady

- **Nie używaj t-SNE/UMAP do redukcji wymiarów przed modelem** — to narzędzia do **wizualizacji**!
- Odległości między klastrami w t-SNE **nie mają znaczenia** (zależy od perplexity)
- Różne uruchomienia dają różne wyniki (niestabilne)
- Użyj PCA do redukcji wymiarów przed modelem

In [ ]:
# t-SNE i UMAP — wizualizacja
from sklearn.manifold import TSNE
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

# Zbiór digits (8x8 obrazki cyfr, 64 wymiary)
digits = load_digits()
X_digits, y_digits = digits.data, digits.target
print(f"Digits: {X_digits.shape} → {len(np.unique(y_digits))} klas")

# PCA → 2D
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_digits)

# t-SNE → 2D
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_digits)

# UMAP → 2D (jeśli dostępny)
try:
 import umap
 reducer = umap.UMAP(n_components=2, random_state=42)
 X_umap = reducer.fit_transform(X_digits)
 has_umap = True
except ImportError:
 has_umap = False
 print("pip install umap-learn (opcjonalnie)")

# Wizualizacja porównawcza
ncols = 3 if has_umap else 2
fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 5))

scatter0 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.7)
axes[0].set_title('PCA')
axes[0].legend(*scatter0.legend_elements(), title="Cyfra", fontsize=7, loc='best')

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.7)
axes[1].set_title('t-SNE (perplexity=30)')

if has_umap:
 axes[2].scatter(X_umap[:, 0], X_umap[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.7)
 axes[2].set_title('UMAP')

plt.suptitle('Porównanie redukcji wymiarów: Digits (64D → 2D)')
plt.tight_layout()
plt.show()

---

## 11. DBSCAN, Hierarchical i Spectral Clustering

### DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

**Idea**: klastry to **gęste regiony** rozdzielone rzadkimi obszarami. Nie wymaga podania $k$.

**Hiperparametry**:
- $\varepsilon$ (eps): promień sąsiedztwa
- $\text{min\_samples}$: minimalna liczba punktów, by uznać region za gęsty

**Typy punktów**:
- **Core point**: ma ≥ min_samples sąsiadów w promieniu $\varepsilon$
- **Border point**: nie jest core, ale jest sąsiadem core point
- **Noise**: nie należy do żadnego klastra (etykieta = -1)

**Zalety DBSCAN**:
- Wykrywa klastry o **dowolnym kształcie** (pierścienie, spirale)
- **Automatycznie** wykrywa outliers (noise)
- Nie wymaga podania $k$

**Wady**:
- Wrażliwy na $\varepsilon$ — trudno dobrać dla danych o zmiennej gęstości
- Nie radzi sobie dobrze z klastrami o różnej gęstości

### Hierarchical Clustering (klasteryzacja hierarchiczna)

Buduje **dendrogram** — drzewo łączone (aglomeracyjne) lub dzielone (dywizyjne).

**Agglomerative** (bottom-up, najczęstszy):
1. Każdy punkt = osobny klaster
2. Łącz dwa najbliższe klastry
3. Powtarzaj aż zostanie 1 klaster
4. Dendogram pozwala wybrać $k$ przez "przecięcie"

**Linkage** (jak mierzyć odległość między klastrami):
| Linkage | Odległość | Kształt klastrów |
|---------|----------|-----------------|
| **Single** | Minimum | Wydłużone (chain effect) |
| **Complete** | Maximum | Kompaktowe |
| **Average** | Średnia | Kompromis |
| **Ward** | Minimalizuje WCSS | Sferyczne (jak K-Means) |

### Spectral Clustering (klasteryzacja spektralna)

Używa **spektrum** (wartości własne) macierzy podobieństwa/Laplacianu grafu.

**Algorytm**:
1. Zbuduj **graf podobieństwa** (np. k-NN graph lub RBF kernel)
2. Oblicz **macierz Laplacianu** grafu: $L = D - W$
 - $W$ — macierz wagowa (similarity), $D$ — macierz stopni
3. Oblicz $k$ **najmniejszych wektorów własnych** $L$ (pomijając $\lambda_0 = 0$)
4. Użyj wektorów własnych jako nowych cech → zastosuj **K-Means**

**Warianty Laplacianu**:
- Unnormalized: $L = D - W$
- Normalized (Shi & Malik): $L_{\text{rw}} = D^{-1}L$
- Normalized (Ng, Jordan): $L_{\text{sym}} = D^{-1/2} L D^{-1/2}$

**Zaleta**: wykrywa klastry o złożonych kształtach, które K-Means by nie znalazł (np. pierścienie).

### Porównanie metod klasteryzacji

| Metoda | Kształt klastrów | K wymagane? | Outliers? | Złożoność |
|--------|-----------------|-------------|-----------|-----------|
| **K-Means** | Sferyczne | Tak | Nie | $O(nkd)$ |
| **DBSCAN** | Dowolne | Nie | Tak | $O(n \log n)$ |
| **Hierarchical** | Różne (linkage) | Opcjonalnie | Nie | $O(n^2 \log n)$ |
| **Spectral** | Dowolne (graph) | Tak | Nie | $O(n^3)$ |

In [ ]:
# === DBSCAN, Hierarchical, Spectral — porównanie ===
from sklearn.cluster import DBSCAN, AgglomerativeClustering, SpectralClustering
from sklearn.datasets import make_moons, make_circles

# Dane: pierścienie i półksiężyce (K-Means sobie nie poradzi!)
X_moons, y_moons = make_moons(n_samples=500, noise=0.08, random_state=42)
X_circles, y_circles = make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=42)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row, (X_data, name) in enumerate([(X_moons, "Moons"), (X_circles, "Circles")]):
 # K-Means
 km = KMeans(n_clusters=2, random_state=42, n_init=10)
 axes[row, 0].scatter(X_data[:, 0], X_data[:, 1], c=km.fit_predict(X_data), cmap='tab10', s=10)
 axes[row, 0].set_title(f'{name}: K-Means')

 # DBSCAN
 db = DBSCAN(eps=0.2, min_samples=5)
 labels_db = db.fit_predict(X_data)
 axes[row, 1].scatter(X_data[:, 0], X_data[:, 1], c=labels_db, cmap='tab10', s=10)
 n_noise = (labels_db == -1).sum()
 axes[row, 1].set_title(f'{name}: DBSCAN (noise={n_noise})')

 # Hierarchical (Ward)
 hc = AgglomerativeClustering(n_clusters=2, linkage='single')
 axes[row, 2].scatter(X_data[:, 0], X_data[:, 1], c=hc.fit_predict(X_data), cmap='tab10', s=10)
 axes[row, 2].set_title(f'{name}: Hierarchical (single)')

 # Spectral
 sc = SpectralClustering(n_clusters=2, affinity='nearest_neighbors', random_state=42)
 axes[row, 3].scatter(X_data[:, 0], X_data[:, 1], c=sc.fit_predict(X_data), cmap='tab10', s=10)
 axes[row, 3].set_title(f'{name}: Spectral')

plt.suptitle('K-Means nie radzi sobie z klastrami o złożonym kształcie!', fontsize=13)
plt.tight_layout()
plt.show()

# Dendrogram
from scipy.cluster.hierarchy import dendrogram, linkage

# Mały zbiór dla czytelności
X_small = X_moons[:50]
Z = linkage(X_small, method='ward')

plt.figure(figsize=(12, 4))
dendrogram(Z, truncate_mode='lastp', p=20)
plt.title('Dendrogram (Ward linkage, moons[:50])')
plt.xlabel('Cluster')
plt.ylabel('Distance')
plt.show()

---
---
## Zadania do samodzielnego rozwiązania

---

### Zadanie 1: Metryki od zera (średnie)

Napisz **własne** implementacje (bez sklearn):
- `my_accuracy(y_true, y_pred)`
- `my_precision(y_true, y_pred)` (binarna)
- `my_recall(y_true, y_pred)` (binarna)
- `my_f1(y_true, y_pred)` (binarna)

Sprawdź wyniki z `sklearn.metrics`.

In [ ]:
# Zadanie 1: Metryki od zera
y_true = np.array([1, 1, 0, 1, 0, 0, 1, 1, 0, 1])
y_pred = np.array([1, 0, 0, 1, 0, 1, 1, 1, 0, 0])

# TWÓJ KOD TUTAJ

### Zadanie 2: Algorytm k-NN od zera (trudne)

Zaimplementuj **własny** klasyfikator k-NN:
1. Oblicz odległość euklidesową do wszystkich punktów treningowych
2. Znajdź k najbliższych sąsiadów
3. Zwróć najczęstszą klasę wśród sąsiadów

Porównaj wyniki ze `sklearn.neighbors.KNeighborsClassifier`.

In [ ]:
# Zadanie 2: KNN od zera
class MyKNN:
 def __init__(self, k=5):
 self.k = k

 def fit(self, X, y):
 # TWÓJ KOD TUTAJ
 pass

 def predict(self, X):
 # TWÓJ KOD TUTAJ
 pass

# Test na Iris
my_knn = MyKNN(k=5)
my_knn.fit(X_train, y_train)
my_pred = my_knn.predict(X_val)
print(f"My KNN accuracy: {accuracy_score(y_val, my_pred):.3f}")

### Zadanie 3: Wpływ standaryzacji na KNN (średnie)

1. Wygeneruj dane ze `sklearn.datasets.make_classification` (2 cechy, 200 próbek)
2. Pomnóż jedną cechę przez 1000 (symulacja różnych skali)
3. Trenuj KNN **bez** standaryzacji → accuracy
4. Trenuj KNN **ze** standaryzacją → accuracy
5. Narysuj oba przypadki (scatter z granicami decyzji)

In [ ]:
# Zadanie 3: Wpływ standaryzacji
np.random.seed(42)
# TWÓJ KOD TUTAJ

### Zadanie 4: Klasteryzacja K-Means od zera (trudne)

Zaimplementuj algorytm K-Means:
1. Losowo zainicjuj k centroidów
2. Przypisz każdy punkt do najbliższego centroidu
3. Przelicz centroidy jako średnią przypisanych punktów
4. Powtarzaj aż centroidy się ustabilizują

Wizualizuj wynik na 2D danych.

In [ ]:
# Zadanie 4: K-Means od zera
from sklearn.datasets import make_blobs

X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, random_state=42)

def my_kmeans(X, k, max_iter=100):
 # TWÓJ KOD TUTAJ
 pass

# labels, centroids = my_kmeans(X_blobs, k=3)
# Wizualizacja...

### Zadanie 5: ROC i AUC (trudne)

1. Wygeneruj binarny zbiór danych (`make_classification`)
2. Trenuj 3 modele: KNN (k=1), KNN (k=10), KNN (k=50)
3. Dla każdego modelu oblicz `predict_proba` na zbiorze testowym
4. Narysuj krzywe ROC wszystkich 3 modeli na jednym wykresie
5. Oblicz AUC dla każdego

**ROC (Receiver Operating Characteristic):** oś Y = True Positive Rate (Recall), oś X = False Positive Rate.

In [ ]:
# Zadanie 5: ROC i AUC
np.random.seed(42)
# TWÓJ KOD TUTAJ

### Zadanie 6: Learning curves (srednie)

Narysuj **learning curves** (krzywe uczenia) dla KNN (k=5) na zbiorze Iris:
1. Trenuj na coraz wiekszych podzbiorach treningowych (10%, 20%, ..., 100%)
2. Dla kazdego rozmiaru oblicz accuracy na zbiorze treningowym i walidacyjnym
3. Narysuj obie krzywe na jednym wykresie
4. Zinterpretuj: czy model ma problem z bias czy variance?

Uzyj `sklearn.model_selection.learning_curve`.

In [ ]:
# Zadanie 6: Learning curves
from sklearn.model_selection import learning_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt
import numpy as np
# TWOJ KOD TUTAJ

### Zadanie 7: Porownanie scalerow (srednie)

Na zbiorze z outlierami (np. wygeneruj dane z kilkoma wartosciami odstajacymi):
1. Zastosuj 4 metody skalowania: StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler
2. Dla kazdej metody narysuj histogram przeskalowanych danych
3. Ktory scaler jest najodporniejszy na outliery?
4. Wytrenuj KNN z kazdym scalerem — porownaj accuracy.

In [ ]:
# Zadanie 7: Porownanie scalerow
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
# TWOJ KOD TUTAJ

### Zadanie 8: Metryki na niezbalansowanym zbiorze
Stwórz syntetyczny zbiór binarny z proporcją klas 95:5 (1000 próbek, 20 cech).
Wytrenuj LogisticRegression i oblicz: accuracy, precision, recall, F1, balanced accuracy, ROC AUC.
Porównaj z klasyfikatorem, który zawsze przewiduje klasę większościową.
Która metryka najlepiej odzwierciedla jakość na niezbalansowanych danych?

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, roc_auc_score

# TWOJ KOD TUTAJ


### Zadanie 9: PCA — wyjaśniona wariancja i rekonstrukcja
Na zbiorze digits (sklearn) zastosuj PCA z różną liczbą komponentów (2, 5, 10, 20, 40, 64).
1. Narysuj krzywą skumulowanej wyjaśnionej wariancji (EVR)
2. Ile komponentów potrzeba aby zachować 95% wariancji?
3. Zrekonstruuj przykładowy obraz z 5 i 40 komponentów — porównaj wizualnie

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# TWOJ KOD TUTAJ


---

## Dodatek OAI: Ćwiczenia w stylu olimpiadowym

### Kontekst olimpiadowy
Metryki i ewaluacja to **absolutny fundament** każdego zadania OAI. W każdym zadaniu ocena jest automatyczna i oparta na konkretnej metryce. Przykłady z poprzednich edycji:

- **Balanced Accuracy** → Zaburzenia EKG (II OAI), Zmiany semantyczne (III OAI)
- **F1 macro** → Klasyfikacja wieloetykietowa (III OAI)
- **ROC AUC** → Halucynacje LLM (II OAI)
- **PSNR** → Odszumianie obrazów (II OAI), Inpainting (II OAI)
- **Accuracy** → Większość zadań klasyfikacyjnych

### Ćwiczenie OAI-5A: Funkcja punktująca (wzorzec olimpiadowy)

Napisz funkcję `compute_score(metric_value)` wzorowaną na III OAI (zmiany semantyczne):
- Balanced Accuracy ≤ 0.7 → 0 punktów
- Balanced Accuracy ≥ 0.87 → 100 punktów
- Pomiędzy: skalowanie liniowe

### Ćwiczenie OAI-5B: Porównanie modeli pod kątem czasu

Na olimpiadzie masz **limit czasowy** (zwykle 1-15 minut). Wytrenuj 3 modele na zbiorze Iris (KNN, SVM, Random Forest), zmierz czas treningu + predykcji, i wybierz najlepszy kompromis jakość/czas.

Użyj `time.time()` do pomiaru.

### Ćwiczenie OAI-5C: Cross-validation z ograniczeniami

Zaimplementuj 5-fold CV na dowolnym zbiorze, ale z ograniczeniem: cały pipeline (trening + ewaluacja) musi zmieścić się w 30 sekundach. Raportuj balanced accuracy i czas.

In [ ]:
import numpy as np
import time
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score

# === Ćwiczenie OAI-5A: Funkcja punktująca ===
def compute_score(bal_acc: float, threshold_low=0.7, threshold_high=0.87) -> int:
 """
 Wzorzec z III OAI (zmiany semantyczne).
 Skalowanie liniowe między progami.
 """
 if bal_acc <= threshold_low:
 return 0
 elif bal_acc >= threshold_high:
 return 100
 else:
 return int(round(100 * (bal_acc - threshold_low) / (threshold_high - threshold_low)))

# Test
for acc in [0.5, 0.7, 0.75, 0.80, 0.87, 0.95]:
 print(f" Balanced Accuracy = {acc:.2f} → {compute_score(acc)} pkt")

# === Ćwiczenie OAI-5B: Porównanie modeli z limitem czasu ===
print("\n=== Porównanie modeli (jakość vs czas) ===")
X, y = make_classification(n_samples=5000, n_features=20, n_classes=3, 
 n_informative=10, random_state=42)

models = {
 'KNN(k=5)': KNeighborsClassifier(n_neighbors=5),
 'SVM(rbf)': SVC(kernel='rbf', random_state=42),
 'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
 start = time.time()
 scores = cross_val_score(model, X, y, cv=5, scoring='balanced_accuracy')
 elapsed = time.time() - start
 print(f" {name:15s} | bal_acc = {scores.mean():.4f} ± {scores.std():.4f} | czas = {elapsed:.2f}s")

# Ćwiczenie OAI-5C: Twoja kolej — zaimplementuj CV z limitem 30 sekund
# TODO: Twój kod tutaj
print("\n Wskazówka OAI: Na olimpiadzie zawsze mierz czas — przekroczenie limitu = 0 punktów!")